# MambaFold EqM — 3D Structure Visualization

**세 가지 inference 비교:**
- **1-step**: γ 위치에서 forward pass 1회 → x_hat
- **EqM Euler**: 확률흐름 ODE 수치 적분, `dx/dγ = (x_hat - x)/(1-γ)`
- **EqM NAG**: Nesterov lookahead + Euler, `y_k = x_k + μ(x_k - x_{k-1})`

```
outputs/overfit/<job_id>/inference.npz  ← GPU에서 자동 저장, 노트북은 로드만
```

In [ ]:
%matplotlib widget
# %matplotlib inline   ← interactive 안될 때

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from pathlib import Path
print("imports ok")

## Config

In [ ]:
JOB_ID = "22627"
REPO   = Path("../")

NPZ = REPO / f"outputs/overfit/{JOB_ID}/inference.npz"
print(f"NPZ: {NPZ}  exists={NPZ.exists()}")

## 1. 데이터 로드

In [ ]:
data = np.load(NPZ)
print("Keys:", sorted(data.keys()))

# single-step arrays
x_clean  = data["x_clean_ca"]           # [G, S, L, 3] Å
x_noisy  = data["x_noisy_ca"]           # [G, S, L, 3]
x_hat    = data["x_hat_ca"]             # [G, S, L, 3]
ca_mask  = data["ca_mask"].astype(bool) # [L]
gammas   = data["gammas"]               # [G]
rmsds    = data["rmsds"]                # [G, S]
lddts    = data["lddts"]                # [G, S]

# EqM Euler
has_euler = "euler_final_ca" in data
if has_euler:
    euler_final = data["euler_final_ca"]  # [DS, L, 3]
    euler_traj  = data["euler_traj_ca"]   # [DS, T, L, 3]
    euler_sched = data["euler_gammas"]    # [T+1]
    euler_rmsd  = data["euler_rmsd"]      # [DS]
    DS, T, L, _ = euler_traj.shape

# EqM NAG
has_nag = "nag_final_ca" in data
if has_nag:
    nag_final = data["nag_final_ca"]  # [DS, L, 3]
    nag_traj  = data["nag_traj_ca"]   # [DS, T, L, 3]
    nag_sched = data["nag_gammas"]
    nag_rmsd  = data["nag_rmsd"]      # [DS]

G, S, L, _ = x_clean.shape
print(f"Single-step: G={G}, S={S}, L={L}")
if has_euler: print(f"Euler: DS={DS}, T={T}")
print(f"NAG: {has_nag}")

## 2. 단일 γ — 1-step 3D 인터랙티브

In [ ]:
def plot_3d_1step(gi, si=0, ax=None, show_arrows=True, arrow_every=8):
    if ax is None:
        fig = plt.figure(figsize=(9, 7))
        ax = fig.add_subplot(111, projection="3d")
    mask = ca_mask
    g, rmsd = gammas[gi], rmsds[gi, si]
    for arr, color, label, lw, ms in [
        (x_clean[gi, si], "#27ae60", "Clean (GT)",            2.5, 30),
        (x_noisy[gi, si], "#95a5a6", f"Noisy γ={g:.2f}",     1.0, 10),
        (x_hat[gi, si],   "#e74c3c", "1-step reconstruction", 2.0, 20),
    ]:
        pts = arr[mask]
        ax.plot(*pts.T, color=color, label=label, lw=lw, alpha=0.85)
        ax.scatter(*pts.T, color=color, s=ms, alpha=0.7, depthshade=False)
    if show_arrows:
        xh, xc = x_hat[gi, si][mask], x_clean[gi, si][mask]
        for i in range(0, len(xh), arrow_every):
            d = xc[i] - xh[i]
            ax.quiver(*xh[i], *d, color="#f39c12", alpha=0.5,
                      arrow_length_ratio=0.3, linewidth=0.9)
    ax.set_title(f"1-step  γ={g:.2f}   RMSD={rmsd:.2f} Å", fontsize=11)
    ax.legend(fontsize=8); ax.set_xlabel("X(Å)"); ax.set_ylabel("Y(Å)"); ax.set_zlabel("Z(Å)")
    return ax

hi = np.argmin(np.abs(gammas - 0.91))
fig = plt.figure(figsize=(9, 7))
plot_3d_1step(hi, ax=fig.add_subplot(111, projection="3d"))
plt.tight_layout(); plt.show()

## 3. EqM Euler 궤적 — noise → structure

In [ ]:
if has_euler:
    SI = 0
    gt = x_clean[0, 0]
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    cmap = plt.cm.Blues
    n_show = min(6, T)
    for k, ti in enumerate(np.linspace(0, T-1, n_show, dtype=int)):
        pts = euler_traj[SI, ti][ca_mask]
        c = cmap(0.3 + 0.7 * k / max(n_show-1, 1))
        ax.plot(*pts.T, color=c, lw=1.2, alpha=0.6,
                label=f"step {ti} (γ={euler_sched[ti]:.2f})")

    ax.plot(*euler_final[SI][ca_mask].T, color="#e74c3c", lw=2.5,
            label=f"Final  RMSD={euler_rmsd[SI]:.2f} Å")
    ax.plot(*gt[ca_mask].T, color="#27ae60", lw=2.5, ls="--", label="GT")
    ax.set_title(f"EqM Euler trajectory  ({T} steps, seed {SI})", fontsize=11)
    ax.legend(fontsize=7, loc="upper left")
    ax.set_xlabel("X(Å)"); ax.set_ylabel("Y(Å)"); ax.set_zlabel("Z(Å)")
    plt.tight_layout(); plt.show()

## 4. EqM NAG 궤적 — noise → structure

In [ ]:
if has_nag:
    SI = 0
    gt = x_clean[0, 0]
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")

    cmap = plt.cm.Purples
    n_show = min(6, T)
    for k, ti in enumerate(np.linspace(0, T-1, n_show, dtype=int)):
        pts = nag_traj[SI, ti][ca_mask]
        c = cmap(0.3 + 0.7 * k / max(n_show-1, 1))
        ax.plot(*pts.T, color=c, lw=1.2, alpha=0.6,
                label=f"step {ti} (γ={nag_sched[ti]:.2f})")

    ax.plot(*nag_final[SI][ca_mask].T, color="#9b59b6", lw=2.5,
            label=f"Final  RMSD={nag_rmsd[SI]:.2f} Å")
    ax.plot(*gt[ca_mask].T, color="#27ae60", lw=2.5, ls="--", label="GT")
    ax.set_title(f"EqM NAG trajectory  (μ=0.9, {T} steps, seed {SI})", fontsize=11)
    ax.legend(fontsize=7, loc="upper left")
    ax.set_xlabel("X(Å)"); ax.set_ylabel("Y(Å)"); ax.set_zlabel("Z(Å)")
    plt.tight_layout(); plt.show()

## 5. 직접 비교: 1-step vs Euler vs NAG

In [ ]:
if has_euler and has_nag:
    gt   = x_clean[0, 0]
    hi_g = np.argmin(np.abs(gammas - 0.9))
    fig  = plt.figure(figsize=(16, 6))

    for pos, (arr, color, title) in enumerate([
        (x_hat[hi_g, 0],  "#e74c3c", f"1-step (γ=0.9)\nRMSD={rmsds[hi_g,0]:.2f} Å"),
        (euler_final[0],  "#3498db", f"EqM Euler ({T} steps)\nRMSD={euler_rmsd[0]:.2f} Å"),
        (nag_final[0],    "#9b59b6", f"EqM NAG (μ=0.9, {T} steps)\nRMSD={nag_rmsd[0]:.2f} Å"),
    ], 1):
        ax = fig.add_subplot(1, 3, pos, projection="3d")
        ax.plot(*gt[ca_mask].T,  color="#27ae60", lw=2, label="GT",    alpha=0.9)
        ax.plot(*arr[ca_mask].T, color=color,     lw=2, label="Pred",  alpha=0.9)
        ax.set_title(title, fontsize=10)
        ax.legend(fontsize=7)

    fig.suptitle("1-step vs EqM Euler vs EqM NAG (seed 0)", fontsize=13)
    plt.tight_layout()
    out = REPO / "outputs" / "viz3d_compare.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    print(f"Saved: {out}")
    plt.show()

## 6. RMSD vs γ (1-step) + Euler/NAG 수평선

In [ ]:
rmsd_mean = rmsds.mean(1); rmsd_std = rmsds.std(1)
lddt_mean = lddts.mean(1); lddt_std = lddts.std(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(gammas, rmsd_mean, color="#e74c3c", lw=2, label="1-step RMSD")
ax1.fill_between(gammas, rmsd_mean-rmsd_std, rmsd_mean+rmsd_std,
                 alpha=0.2, color="#e74c3c")
if has_euler:
    ax1.axhline(euler_rmsd.mean(), ls="--", color="#3498db", lw=1.5,
                label=f"Euler mean {euler_rmsd.mean():.2f} Å")
if has_nag:
    ax1.axhline(nag_rmsd.mean(), ls="--", color="#9b59b6", lw=1.5,
                label=f"NAG mean {nag_rmsd.mean():.2f} Å")
ax1.axhline(2.0, ls=":", color="gray", lw=1)
ax1.set_xlabel("γ"); ax1.set_ylabel("Cα-RMSD (Å)")
ax1.set_title("RMSD: 1-step vs Euler vs NAG")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.plot(gammas, lddt_mean, color="#27ae60", lw=2, label="1-step LDDT")
ax2.fill_between(gammas, lddt_mean-lddt_std, lddt_mean+lddt_std,
                 alpha=0.2, color="#27ae60")
ax2.axhline(0.7, ls=":", color="gray", lw=1)
ax2.set_ylim(0, 1); ax2.set_xlabel("γ"); ax2.set_ylabel("Hard LDDT")
ax2.set_title("LDDT vs γ (1-step)")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 7. Euler/NAG trajectory step 슬라이더

In [ ]:
if has_euler and has_nag:
    try:
        from ipywidgets import interact, IntSlider, Dropdown

        gt  = x_clean[0, 0]
        fig_s, ax_s = plt.subplots(figsize=(7, 7))

        def update_step(step=T-1, sampler="Euler", seed=0):
            ax_s.cla()
            traj = euler_traj if sampler == "Euler" else nag_traj
            sched = euler_sched if sampler == "Euler" else nag_sched
            color = "#3498db" if sampler == "Euler" else "#9b59b6"
            pts_gt   = gt[ca_mask]
            pts_step = traj[seed, step][ca_mask]
            ax_s.plot(*pts_gt[:, :2].T,   "g-o", ms=3, lw=2, label="GT")
            ax_s.plot(*pts_step[:, :2].T, color=color, marker="o", ms=3, lw=2,
                      label=f"{sampler} step {step}  γ={sched[step]:.2f}")
            rmsd_step = np.sqrt(((pts_step - pts_gt)**2).sum(-1).mean())
            ax_s.set_title(f"{sampler} step {step}/{T-1}  RMSD={rmsd_step:.2f} Å")
            ax_s.set_aspect("equal"); ax_s.legend(fontsize=9); ax_s.grid(alpha=0.3)
            fig_s.canvas.draw_idle()

        interact(update_step,
                 step=IntSlider(min=0, max=T-1, step=1, value=T-1,
                                description="step", continuous_update=False),
                 sampler=Dropdown(options=["Euler", "NAG"], description="sampler"),
                 seed=IntSlider(min=0, max=DS-1, step=1, value=0, description="seed"))
    except ImportError:
        print("ipywidgets 없음")

## 8. 1-step slider (γ 선택)

In [ ]:
try:
    from ipywidgets import interact, IntSlider, Checkbox
    fig_i = plt.figure(figsize=(9, 7))
    ax_i  = fig_i.add_subplot(111, projection="3d")

    def update_1step(gamma_idx=hi, seed=0, arrows=True):
        ax_i.cla()
        plot_3d_1step(gamma_idx, si=seed, ax=ax_i, show_arrows=arrows)
        fig_i.canvas.draw_idle()

    interact(update_1step,
             gamma_idx=IntSlider(min=0, max=G-1, step=1, value=hi,
                                 description="γ index", continuous_update=False),
             seed=IntSlider(min=0, max=S-1, step=1, value=0, description="seed"),
             arrows=Checkbox(value=True, description="화살표"))
except ImportError:
    print("ipywidgets 없음")